# Building Makemore
Now we are going to be building a character level model that makes use of "bigrams" which are basically just pairs of words. We will train this character level language model to makemore of human names taken from the SSA, specifically 

"...The included names.txt dataset, as an example, has the most common 32K names takes from ssa.gov for the year 2018." (From https://github.com/karpathy/makemore)

We begin by showing an example of the bigrams in the first name after adding start and end tokens.

In [ ]:
with open("names.txt") as f:
    print(f.readline()) # show a single name from the list of 32k names
    all_names = [name.rstrip('\n') for name in f.readlines()] 

# print(all_names)

name1 = ['<s>'] + list(all_names[0]) + ['<e>']
print(name1)

for c1, c2 in zip(name1, name1[1:]):
    print(c1,c2)

# Making a Bigram Dictionary
We're now going to use these names to examine statistical properties of the bigrams that exist in this dataset, namely the number of occurences.

In [ ]:
bigram_dict = {}

for name in all_names:
    name_with_special_tokens = ['<s>'] + list(name) + ['<e>']
    for c1, c2 in zip(name_with_special_tokens, name_with_special_tokens[1:]):
        bigram = (c1,c2)
        bigram_dict[bigram] = bigram_dict.get(bigram, 0) + 1 # increment the count of this specific bigram

print(bigram_dict)

In [ ]:
# Which bigrams were the most frequent?

# items() turns this into a list of [ ((c1,c2), counts), ... ]
# then we used the sorted key param to organize this by the counts instead of the tuple
sorted(bigram_dict.items(), key=lambda kv: -kv[1])

Now we see the most common bigrams are `n<e>`, `a<e>`, and `an`. It is very common for a name
to end with `n` or `a`. Because we are going to be using `pytorch` it will be much easier if we convert this into a tensor instead of a python dictionary. We do that below.

In [ ]:
all_names_as_str = ''.join(all_names) # combine all names into one string
all_letters = set(all_names_as_str) # remove dupes
list_of_letters = sorted(list(all_letters))

# Now make a mapping so we can represent our data as a 2x2 tensor with the counts as the cell entries
stoi = {letter: interger_representation for interger_representation, letter in enumerate(list_of_letters)}
stoi['<s>'] = 26
stoi['<e>'] = 27
stoi

In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
N = torch.zeros(len(stoi),len(stoi), dtype=torch.int32)

for name in all_names:
    name_with_special_tokens = ['<s>'] + list(name) + ['<e>']
    for c1, c2 in zip(name_with_special_tokens, name_with_special_tokens[1:]):
        N[stoi[c1], stoi[c2]] += 1 # Increment the cell for this bigram lookup in the tensor

print(N[stoi['a'], stoi['n']])
print(bigram_dict[('a', 'n')])
        
# They match! Let's visualize

In [ ]:
itos = {s:i for i,s in stoi.items()}

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(N.shape[0]):
    for j in range(N.shape[1]):
        plt.text(i,j, itos[i] + itos[j], ha='center', va='top', color='gray')
        plt.text(i,j,N[i,j].item(), ha='center', va='bottom', color='gray')
plt.axis('off')


In [ ]:
N[1,1].item()

# Wasting Space
One issue we can immediately see is that we have an entire row and column of combinations that can never happen, namely something following `<e>`, and something preceding `<s>`. To clean this up, Andrej recommends just calling the tokens both `.` and redoing the mappings. This basically flips the meaning of `.b` which typically would have indicated something starting after `<e>`, to mean that the first letter was `b`. Another way to think about it is adding `.b` twice, one for the combination of "`b` comes after the end of token, and `b` comes after the start token which is 0 + some amount.

In [ ]:
# Now make a mapping so we can represent our data as a 2x2 tensor with the counts as the cell entries
stoi = {letter: interger_representation+1 for interger_representation, letter in enumerate(list_of_letters)}
stoi['.'] = 0
itos = {s:i for i,s in stoi.items()}

N = torch.zeros(len(stoi),len(stoi), dtype=torch.int32)

for name in all_names:
    name_with_special_tokens = ['.'] + list(name) + ['.']
    for c1, c2 in zip(name_with_special_tokens, name_with_special_tokens[1:]):
        N[stoi[c1], stoi[c2]] += 1 # Increment the cell for this bigram lookup in the tensor

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(N.shape[0]):
    for j in range(N.shape[1]):
        plt.text(i,j, itos[i] + itos[j], ha='center', va='top', color='gray')
        plt.text(i,j,N[i,j].item(), ha='center', va='bottom', color='gray')
plt.axis('off')

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p

In [ ]:

# We need to now sample from our bigrams to start generating plausible combinations
# probabilistically.

# Get the first row and get the distribution of those bigrams for a starting character
bigram_p = N[0].float()
bigram_p = bigram_p / bigram_p.sum()
print(bigram_p)

# Now we can select the first letter in the name randomly
g = torch.Generator().manual_seed(2147483647)
first_letter_idx = torch.multinomial(bigram_p, 1, replacement=True, generator=g).item()
print(first_letter_idx)
print(itos[first_letter_idx])



# Generating a Name
Now that we understand how to statistically sample the bigrams, we can continue the "chain" and keep producing more characters from our frequency table until we hit an end of character `.` token. 

In [ ]:

for k in range(20):
    c_idx = 0
    c = None
    name = ''
    while c != '.':
        # While we aren't at the end token, find the distribution corresponding to the current character
        # print(f'Character is "{itos[c_idx]}", therefore we must sample from row {c_idx}')
        p = N[c_idx]
        p = p / p.sum()

        # Then sample from this distribution
        c_idx = torch.multinomial(p, 1, replacement=True, generator=g).item()
        
        # look up the character and set it for next time, this will tell us which distribution to look at
        c = itos[c_idx]
        name += c

    name = name.rstrip('.')
    print(name)

# Weird Names
It may look like our model is not working, but in fact this is just the weakness of the bigram method. There are some names like `j.` which typically wouldn't make sense in a normal name, but the bigram model does not care about what came before the `j`, it only generates the next token according to which bigrams that use `j` are likely, and `j.` is one possibility. We can compare our statistically "trained" bigram language model to an "untrained" model by making all bigrams equally likely across each row.

In [ ]:

for k in range(20):
    c_idx = 0
    c = None
    name = ''
    while c != '.':
        # While we aren't at the end token, find the distribution corresponding to the current character
        # print(f'Character is "{itos[c_idx]}", therefore we must sample from row {c_idx}')
        p = torch.ones(27)
        p = p / p.sum()

        # Then sample from this distribution
        c_idx = torch.multinomial(p, 1, replacement=True, generator=g).item()
        
        # look up the character and set it for next time, this will tell us which distribution to look at
        c = itos[c_idx]
        name += c

    name = name.rstrip('.')
    print(name)

We can see here that this untrained model does in fact perform much worse, and they have no resemblence to any names whatsoever. Therefore even though our bigram model isn't the best, it's still a stark improvement over random character generations.